# Inspect an object's loaded metadata

Print a short description, request the full summary, and inspect incomplete data.
This notebook executes the handbook's `BuildingSummary` example. It is not the
future EPSM Building API and performs no geometry calculation or simulation.

**Before starting:** Python 3.11 or newer; no extra packages. Start the kernel in
this repository or its `examples/` directory. The fixture source ID is
`synthetic.handbook.object-inspection.v1`. Counts are dimensionless. No random
sampling or seed is used. Expected outcome: all assertions pass; no files are
written by object creation or inspection.


## Load the checked documentation template

The setup reads the repository template explicitly. `.tmpl` files are documentation
resources, so they are loaded with Python's standard `runpy` module. A package's
own examples should import its installed public API instead.


In [ ]:
from pathlib import Path
import runpy

repo_root = Path.cwd().resolve()
if repo_root.name == "examples":
    repo_root = repo_root.parent

template_path = repo_root / "docs/templates/public-entity.py.tmpl"
if not template_path.is_file():
    raise FileNotFoundError("Open this notebook from the repository root or examples folder")

template = runpy.run_path(str(template_path))
BuildingSummary = template["BuildingSummary"]


## Start with one loaded record

The record holds an identifier, state and an already known part count. Printing
it reads those values; it does not derive readiness or count geometry parts.


In [ ]:
item = BuildingSummary("example-building", "draft", 2)
print(item)

summary = item.summary()
assert summary == {"identifier": "example-building", "state": "draft", "part_count": 2}
summary


## Keep an unknown count visible

`None` means the count has not been loaded. It does not mean zero parts or a failed
simulation. The complete summary retains the unknown value.


In [ ]:
unknown = BuildingSummary("example-building", "incomplete", None)
assert "parts=not_loaded" in repr(unknown)
assert unknown.summary()["part_count"] is None
assert str(unknown) == repr(unknown)
print(unknown)


## Check long labels and control characters

A display stays on one printable line within 160 characters, even when labels
contain controls or long Unicode text. Shortening the display leaves stored data
unchanged. The full summary is available when needed.


In [ ]:
identifier = "example\n\t\x00" + "x" * 1_000
state = "🏠" * 1_000
large = BuildingSummary(identifier, state, 2)
description = repr(large)

assert len(description) <= 160
assert description.isprintable()
assert "…" in description
assert large.summary()["identifier"] == identifier
assert large.summary()["state"] == state
print(description)


## Check a fresh run

Use **Restart Kernel and Run All**. All assertions must pass in order. Clear
outputs and execution counts before committing. This demonstrates inspection of
loaded metadata; it does not establish any EPSM modelling or scientific result.
